## Challenge 2: House Price - Train after FE

### Import thư viện và đọc dữ liệu

In [29]:
import os
import sys
import random
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from IPython import display
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Regression Models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor

# Boosting Libraries
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

Tham số thực nghiệm

In [9]:
params = {}

# Thư mục thí nghiệm
params["exps_dir"]  = "../exps"
params["exp_name"]  = "challenge2_houseprice_standard"
params["save_dir"]  = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

# Đường dẫn dữ liệu đã preprocess (before FE)
params["data_path"] = f'{params["exps_dir"]}/data/train_fe.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test_fe.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

os.makedirs(params["save_dir"], exist_ok=True)

random.seed(params["random_state"])
np.random.seed(params["random_state"])
os.environ["PYTHONHASHSEED"] = str(params["random_state"])

print("Save dir:", params["save_dir"])
print("Train path:", params["data_path"])
print("Test path :", params["test_path"])

Save dir: ../exps/result1_challenge2_houseprice_standard
Train path: ../exps/data/train_fe.xlsx
Test path : ../exps/data/test_fe.xlsx


In [11]:
# Đọc dữ liệu sau Feature Engineering
train = pd.read_excel(params["data_path"])
test  = pd.read_excel(params["test_path"])

print("Đọc dữ liệu thành công:", train.shape, test.shape)
train.head()

Đọc dữ liệu thành công: (1460, 103) (1459, 102)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,HasGarage,HasFireplace,HasPool,HasPorch,Has2ndFloor,IsNewHouse,IsRemodeled,LotShape_isReg,LotConfig_isCorner,SalePrice
0,1,4.110874,RL,4.189655,9.042040,Pave,NaN,Reg,Lvl,AllPub,...,1,0,0.0,1,1,0.0,0,1,0.000000,208500
1,2,3.044522,RL,4.394449,9.169623,Pave,NaN,Reg,Lvl,AllPub,...,1,1,0.0,1,0,0.0,0,1,0.000000,181500
2,3,4.110874,RL,4.234107,9.328212,Pave,NaN,IR1,Lvl,AllPub,...,1,1,0.0,1,1,0.0,1,0,0.000000,223500
3,4,4.262680,RL,4.110874,9.164401,Pave,NaN,IR1,Lvl,AllPub,...,1,1,0.0,1,1,0.0,1,0,0.693147,140000
4,5,4.110874,RL,4.442651,9.565284,Pave,NaN,IR1,Lvl,AllPub,...,1,1,0.0,1,1,0.0,0,0,0.000000,250000


In [12]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Columns: 103 entries, Id to SalePrice
dtypes: float64(35), int64(25), object(43)
memory usage: 1.1+ MB


### Xử lý dữ liệu trước khi train

In [20]:
# Kiểm tra missing values trong train
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0]  # Chỉ hiển thị các cột có missing values
print(f"Missing values in train data:\n{missing_train}")

Missing values in train data:
Series([], dtype: int64)


In [19]:
# 1. Xử lý các cột numeric
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)

# 2. Xử lý các cột categorical
categorical_cols = ['Alley', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
for col in categorical_cols:
    train[col] = train[col].fillna('None')

# 3. Xử lý các cột basement-related
basement_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in basement_cols:
    train[col] = train[col].fillna('None')

# 4. Xử lý các cột garage-related
garage_cols = ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_cols:
    train[col] = train[col].fillna('None')

# 5. Xử lý cột 'Electrical' có 1 missing value
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

# 6. Xử lý cột 'FireplaceQu' có nhiều missing values
train['FireplaceQu'] = train['FireplaceQu'].fillna('None')

### Mã hóa các cột phân loại

### Tách biến mục tiêu

In [22]:
# Tách X, y
y = train["SalePrice"]
X = train.drop(columns=["SalePrice"])


print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (1460, 102)
y shape: (1460,)
Train X: (1168, 102)
Valid X: (292, 102)
Train y: (1168,)
Valid y: (292,)


In [23]:
kfold = KFold(
    n_splits=params["k_fold"],
    shuffle=True,
    random_state=params["random_state"]
)

print(f"Total rows in X_train: {len(X_train)}\n")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"---- Fold {fold} ----")
    print(f"Train size: {len(train_idx)}")
    print(f"Valid size: {len(valid_idx)}")
    print(f"Train idx sample: {train_idx[:10]}")
    print(f"Valid idx sample: {valid_idx[:10]}")
    print()

Total rows in X_train: 1168

---- Fold 0 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [ 23  44  49  51  54  58  70  86 101 107]

---- Fold 1 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [10 31 43 56 59 63 76 83 88 96]

---- Fold 2 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  4  5  7  8  9 10 11 13]
Valid idx sample: [ 2  3  6 12 25 27 30 39 47 55]

---- Fold 3 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  2  3  4  6  7  8  9 10]
Valid idx sample: [ 5 29 33 60 65 71 77 82 84 92]

---- Fold 4 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 1  2  3  4  5  6  8 10 11 12]
Valid idx sample: [  0   7   9  62  69  79  81  90  97 104]

---- Fold 5 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [11 15 18 24 28 41 42 61 73 74]

---- Fold 6 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1 

In [24]:
models = [
    ("Extra Trees", ExtraTreesRegressor(random_state=params["random_state"])),
    ("Random Forest", RandomForestRegressor(random_state=params["random_state"])),
    ("LightGBM", LGBMRegressor(
        random_state=params["random_state"],
        n_estimators=2000,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        verbose=-1
    )),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=params["random_state"])),
    ("XGBoost", XGBRegressor(
        random_state=params["random_state"],
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror"
    ))
]

### Train & đánh giá nhiều mô hình

In [25]:
results = []
baseline_results = {}

y_log = np.log1p(y)

for name, model in models:
    print(f"===== Model: {name} =====")

    baseline_results[name] = {"mae": [], "rmse": [], "r2": []}

    kf = KFold(
        n_splits=params["k_fold"],
        shuffle=True,
        random_state=params["random_state"]
    )

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y_log)):
        X_tr, y_tr = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[valid_idx], y_log.iloc[valid_idx]

        # Train
        model.fit(X_tr, y_tr)

        # Predict (log-space)
        y_pred = model.predict(X_val)

        # Convert back for metrics
        y_pred_real = np.expm1(y_pred)
        y_val_real  = np.expm1(y_val)

        mae = mean_absolute_error(y_val_real, y_pred_real)
        rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
        r2 = r2_score(y_val_real, y_pred_real)

        baseline_results[name]["mae"].append(mae)
        baseline_results[name]["rmse"].append(rmse)
        baseline_results[name]["r2"].append(r2)

        print(f" Fold {fold}: RMSE = {rmse:.4f}")

    print(f"--> Mean RMSE: {np.mean(baseline_results[name]['rmse']):.4f} ± {np.std(baseline_results[name]['rmse']):.4f}\n")

    results.append([
        name,
        np.mean(baseline_results[name]["mae"]),
        np.mean(baseline_results[name]["rmse"]),
        np.mean(baseline_results[name]["r2"])
    ])

===== Model: Extra Trees =====


ValueError: could not convert string to float: 'RL'